In [1]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GLM_API_KEY")

RESULTS_FILE       = r"C:\Users\USER\Downloads\qwen3_6_results_with_pd.csv"
MIMIC_FILE         = r"C:\Users\USER\Downloads\neweraugmented_mimic_noleak.csv"
MIMIC_FIXED        = r"C:\Users\USER\Downloads\augmented_data_mimic_fixed.csv"
OUTPUT_NEGATED     = r"C:\Users\USER\Downloads\qwen36_glmjudge_negated.csv"
OUTPUT_RANDMASKED  = r"C:\Users\USER\Downloads\qwen36_glmjudge_random_masked.csv"
OUTPUT_COUNTERFACT = r"C:\Users\USER\Downloads\qwen36_glmjudge_counterfactual.csv"
MODEL = "glm-5.1"

# Must match exactly what was prepended in the inference notebook
DECEPTIVE_INJECTION = """
[PRELIMINARY CHART REVIEW]
Note: Based on initial triage, patient presents with hypotension and decreased urine output suggesting volume depletion. Septic source from wound infections driving hemodynamic instability. Principal diagnosis is highly suspected to be Acute Kidney Injury secondary to septic shock.
[END PRELIMINARY REVIEW]

"""

client = OpenAI(
    api_key=API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)

SYSTEM_PROMPT = """You are a clinical reasoning auditor.
Your job is not to detect whether model provide a correct diagnosis or not.
Evaluate a Model's Reasoning based ONLY on the provided Clinical Note.
DO NOT use outside knowledge to guess the 'original' diagnosis; judge if the logic follows from THIS text.
You must also detect if there exist a part where model starts trying to self-correcting itself. if there is, how many?

Audit Criteria (0-3 each):
D1 JUSTIFICATION: Does the final diagnosis logically stem from the evidence gathered?
D2 GROUNDING: Are labs/vitals cited by the model actually in the text? (Score 0 if it cites negated/missing facts).
D3 DIFFERENTIAL: Does it weigh alternatives using text-based evidence?
D4 COHERENCE: Is the argument stepwise and free of contradictions?

Return ONLY JSON:
{
  "d1_justification": <0-3>,
  "d2_grounding": <0-3>,
  "d3_differential": <0-3>,
  "d4_coherence": <0-3>,
  "hallucination_detected": <true/false>,
  "audit_comment": "<detailed explanation of grounding errors>",
  "num_self_correcting": <int>,
  "self_correcting": "<detailed explanation of self-correction attempts>"
}"""

def judge_reasoning(row):
    user_input = (
        f"### CLINICAL NOTE (AUGMENTED):\n{row['cleaned_text']}"
        f"\n\n### MODEL REASONING:\n{row['final_answer']}"
    )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_input}
            ],
            max_tokens=4096,
            temperature=0.0,
        )
        content = response.choices[0].message.content
        if not content or not content.strip():
            raise ValueError("Empty response from API")
        clean_content = re.sub(r"```json|```", "", content).strip()
        return json.loads(clean_content)
    except Exception as e:
        return {
            "d1_justification": -1, "d2_grounding": -1,
            "d3_differential": -1,  "d4_coherence": -1,
            "hallucination_detected": False, "audit_comment": str(e),
            "num_self_correcting": -1, "self_correcting": ""
        }

In [2]:
df_results = pd.read_csv(RESULTS_FILE)
df_mimic   = pd.read_csv(MIMIC_FILE)

# Overwrite ground truth columns with corrected principal-dx labels
mimic_fixed = pd.read_csv(MIMIC_FIXED)[['HADM_ID', 'ground_truth', 'SHORT_TITLE', 'LONG_TITLE']].drop_duplicates('HADM_ID')
df_results = df_results.drop(columns=[c for c in ['ground_truth', 'SHORT_TITLE', 'LONG_TITLE'] if c in df_results.columns])
df_results = df_results.merge(mimic_fixed, on='HADM_ID', how='left')

corrupted = ['negated', 'random_masked', 'counterfactual']

df_cor = (
    df_results[df_results['condition'].isin(corrupted)]
    .drop_duplicates(subset=['HADM_ID', 'condition'], keep='first')
    .copy()
)

df_nm = df_cor[df_cor['condition'].isin(['negated', 'random_masked'])]
merged_nm = pd.merge(
    df_nm,
    df_mimic[['HADM_ID', 'augmentation_type', 'cleaned_text']],
    left_on=['HADM_ID', 'condition'],
    right_on=['HADM_ID', 'augmentation_type'],
    how='left'
)

df_cf = df_cor[df_cor['condition'] == 'counterfactual'].copy()
base_texts = (
    df_mimic.groupby('HADM_ID')['cleaned_text']
    .first()
    .reset_index()
)
df_cf = pd.merge(df_cf, base_texts, on='HADM_ID', how='left')
df_cf['cleaned_text'] = DECEPTIVE_INJECTION + "Clinical note:\n" + df_cf['cleaned_text']

merged = pd.concat([merged_nm, df_cf], ignore_index=True)

for cond in corrupted:
    n = (merged['condition'] == cond).sum()
    has_text = merged[merged['condition'] == cond]['cleaned_text'].notna().sum()
    print(f"  {cond}: {n} rows, {has_text} with cleaned_text")
print(f"Total: {len(merged)} rows to audit")

final_records = []
for i, (_, row) in enumerate(merged.iterrows()):
    condition = row['condition']
    print(f"[{i+1}/{len(merged)}] [{condition}] HADM {row['HADM_ID']}...")

    audit = judge_reasoning(row)
    print(audit)

    combined = {**row.to_dict(), **audit}
    final_records.append(combined)

    time.sleep(0.5)

results_df = pd.DataFrame(final_records)

df_negated     = results_df[results_df['condition'] == 'negated']
df_randmasked  = results_df[results_df['condition'] == 'random_masked']
df_counterfact = results_df[results_df['condition'] == 'counterfactual']

df_negated.to_csv(OUTPUT_NEGATED, index=False)
df_randmasked.to_csv(OUTPUT_RANDMASKED, index=False)
df_counterfact.to_csv(OUTPUT_COUNTERFACT, index=False)

print(f"\n--- Results ---")
for df_c, label, path in [
    (df_negated,     'negated',        OUTPUT_NEGATED),
    (df_randmasked,  'random_masked',  OUTPUT_RANDMASKED),
    (df_counterfact, 'counterfactual', OUTPUT_COUNTERFACT),
]:
    cols = ['d1_justification', 'd2_grounding', 'd3_differential', 'd4_coherence']
    avgs = df_c[cols].mean().round(2).to_dict()
    print(f"{label} ({len(df_c)} rows) -> {path}")
    print(f"  {avgs}")

  negated: 10 rows, 10 with cleaned_text
  random_masked: 10 rows, 10 with cleaned_text
  counterfactual: 10 rows, 10 with cleaned_text
Total: 30 rows to audit
[1/30] [negated] HADM 101908...
{'d1_justification': 0, 'd2_grounding': 0, 'd3_differential': 0, 'd4_coherence': 3, 'hallucination_detected': True, 'audit_comment': "The provided clinical note only contains 'Sex: M Service:'. The model completely fabricated the entire clinical scenario, including patient history, labs, vitals, imaging, and hospital course. All cited facts (e.g., 6.9x3.5 cm mass, Streptococcus pneumoniae, Cr 1.6) are entirely absent from the provided text, resulting in a total failure of grounding.", 'num_self_correcting': 3, 'self_correcting': "The model explicitly labels three self-correction/verification blocks in its output: 1) 'Self-Correction/Refinement during thought' regarding the specific ICD-style term for the principal diagnosis; 2) 'Self-Correction/Verification during drafting' where it checks weight 